In [3]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import warnings
import os
warnings.filterwarnings('ignore')


In [5]:
distortion_classes = ['Broken Line', 'Edge False Color', 'Over Desaturation', 'Saturated False Color', 'Smears']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
class multi_class_model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1: 50x50 -> 25x25
            nn.Conv2d(3, 20, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 25x25 -> 12x12
            nn.Conv2d(20, 40, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 12x12 -> 6x6
            nn.Conv2d(40, 80, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # 80 filters * 6 * 6 spatial size = 2880
            nn.Linear(80 * 6 * 6, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate and verify
multi_class_model = multi_class_model(len(distortion_classes)).to(device)
print(f"Total parameters: {sum(p.numel() for p in multi_class_model.parameters()):,}")

Total parameters: 406,093


In [11]:
multi_class_model.load_state_dict(torch.load('multi_class_model_weights.pth', map_location=device))

<All keys matched successfully>

In [12]:

# 3. Set to evaluation mode (disables dropout/batchnorm updates)
multi_class_model.to(device)
multi_class_model.eval()

# 4. Define the same transformation used during your validation phase
val_transform = transforms.Compose([
    transforms.Resize((50, 50)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

distortion_classes = ['Broken Line', 'Edge False Color', 'Over Desaturation', 'Saturated False Color', 'Smears']

In [15]:
def classify_distortion(patch_image):
    """Runs a distorted 50x50 PIL image patch through the 5-class model."""
    img_tensor = val_transform(patch_image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = multi_class_model(img_tensor)
        _, pred = torch.max(outputs, 1)
    return distortion_classes[pred.item()]

In [29]:
image_path = "/content/1043739.png"

try:
    img = Image.open(image_path).convert("RGB")
    input_tensor = val_transform(img).unsqueeze(0).to(device) # [1, 3, 50, 50]


    with torch.no_grad():
        output = multi_class_model(input_tensor)

        # Get probabilities using Softmax (optional but helpful for confidence)
        probabilities = torch.softmax(output, dim=1)

        _, predicted = torch.max(output, 1)

        print(f"Raw output (logits): {output}")
        print(f"Confidence: {probabilities[0][predicted.item()].item():.2%}")
        print(f"Predicted class index: {predicted.item()}")
        print(f"Predicted class: {distortion_classes[predicted.item()]}")

except FileNotFoundError:
    print(f"Error: The image at {image_path} was not found.")

Raw output (logits): tensor([[-2.5049,  0.5958, -9.5334, -9.1936,  3.5664]])
Confidence: 94.91%
Predicted class index: 4
Predicted class: Smears


In [33]:
import torch.nn.functional as F

image_path = "/content/56640.png"

try:
    img = Image.open(image_path).convert("RGB")
    input_tensor = val_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = multi_class_model(input_tensor)

        # 1. Convert logits to probabilities (0-1)
        probabilities = torch.softmax(output, dim=1)[0]

        # 2. Get the winner
        conf, predicted = torch.max(probabilities, 0)

        print(f"--- Analysis for: {image_path.split('/')[-1]} ---")
        print(f"PREDICTION: {distortion_classes[predicted.item()]} ({conf.item():.2%})\n")

        print("FULL STATISTICS:")
        # 3. Loop through all classes to see the percentage for each
        for i, class_name in enumerate(distortion_classes):
            prob = probabilities[i].item()
            # We use a simple bar visualization to make it easier to read
            bar = "█" * int(prob * 20)
            print(f"{class_name:22}: {prob:>7.2%} {bar}")

except FileNotFoundError:
    print(f"Error: The image at {image_path} was not found.")

--- Analysis for: 56640.png ---
PREDICTION: Smears (68.90%)

FULL STATISTICS:
Broken Line           :  16.71% ███
Edge False Color      :   5.74% █
Over Desaturation     :   5.44% █
Saturated False Color :   3.21% 
Smears                :  68.90% █████████████


In [34]:
!unzip /content/unbalanced_artifacts_dataset.zip -d /content/data

Streaming output truncated to the last 5000 lines.
  inflating: /content/data/Broken Line/1664755.png  
  inflating: /content/data/Broken Line/7972298.png  
  inflating: /content/data/Broken Line/1665520.png  
  inflating: /content/data/Broken Line/3458048.png  
  inflating: /content/data/Broken Line/3742038.png  
  inflating: /content/data/Broken Line/3322215.png  
  inflating: /content/data/Broken Line/217612.png  
  inflating: /content/data/Broken Line/1574321.png  
  inflating: /content/data/Broken Line/1664590.png  
  inflating: /content/data/Broken Line/9052847.png  
  inflating: /content/data/Broken Line/1665105.png  
  inflating: /content/data/Broken Line/1982566.png  
  inflating: /content/data/Broken Line/3742045.png  
  inflating: /content/data/Broken Line/3738891.png  
  inflating: /content/data/Broken Line/1672242.png  
  inflating: /content/data/Broken Line/4886081.png  
  inflating: /content/data/Broken Line/4884086.png  
  inflating: /content/data/Broken Line/3660267.pn

In [39]:
import torch
import os
from PIL import Image
from torchvision import transforms
from collections import Counter

# 1. Configuration
folder_path = "/content/data/Saturated False Color" # Change to your folder path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class names must match the order in your notebook
distortion_classes = [
    'Broken Line',
    'Edge False Color',
    'Over Desaturation',
    'Saturated False Color',
    'Smears'
]

# 2. Preprocessing (matching your validation transform)
val_transform = transforms.Compose([
    transforms.Resize((50, 50)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 3. Batch Inference Loop
predictions = []
image_files = [f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg'))]

print(f"🔄 Processing {len(image_files)} images...")

multi_class_model.eval() # Ensure model is in eval mode
with torch.no_grad():
    for filename in image_files:
        img_path = os.path.join(folder_path, filename)
        img = Image.open(img_path).convert("RGB")
        input_tensor = val_transform(img).unsqueeze(0).to(device)

        output = multi_class_model(input_tensor)
        _, predicted = torch.max(output, 1)

        predictions.append(distortion_classes[predicted.item()])

# 4. Results Aggregation
counts = Counter(predictions)
total = len(predictions)

print("\n" + "="*30)
print("📊 BATCH TEST RESULTS (Target: Smears)")
print("="*30)

for cls in distortion_classes:
    count = counts.get(cls, 0)
    percentage = (count / total) * 100
    status = "✅ CORRECT" if cls == "Smears" else "❌ MISCLASSIFIED"

    # Create a simple visual bar
    bar = "█" * int(percentage / 5)
    print(f"{cls:22}: {count:>4} ({percentage:>6.2f}%) {bar} {status}")

print("-" * 30)
print(f"TOTAL IMAGES TESTED: {total}")
print(f"ACCURACY FOR SMEARS: {counts.get('Smears', 0)/total:.2%}")

🔄 Processing 1405 images...

📊 BATCH TEST RESULTS (Target: Smears)
Broken Line           :   14 (  1.00%)  ❌ MISCLASSIFIED
Edge False Color      :   55 (  3.91%)  ❌ MISCLASSIFIED
Over Desaturation     :   52 (  3.70%)  ❌ MISCLASSIFIED
Saturated False Color : 1260 ( 89.68%) █████████████████ ❌ MISCLASSIFIED
Smears                :   24 (  1.71%)  ✅ CORRECT
------------------------------
TOTAL IMAGES TESTED: 1405
ACCURACY FOR SMEARS: 1.71%
